# Weather Temperature Prediction with PySpark MLlib

A scalable machine learning pipeline for predicting daily minimum temperatures using Gradient Boosted Trees with hyperparameter tuning.

**Key Features:**
- Time-series feature engineering (lagged temperatures, rolling averages)
- Cross-validation with hyperparameter grid search
- Proper train/test split to avoid data leakage

In [4]:
# install pyspark
!pip install -q pyspark

In [5]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Pyspark_weather_prediction_pipeline") \
    .getOrCreate()

In [6]:
# mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


## 1. Data Preparation

In [7]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Load data
daily_weather = spark.read.parquet(
    'file:/content/gdrive/MyDrive/Big_Data/Pyspark_weather_prediction_pipeline/data/daily_weather.parquet'
)
cities = spark.read.csv(
    'file:/content/gdrive/MyDrive/Big_Data/Pyspark_weather_prediction_pipeline/data/cities.csv',
    header=True, inferSchema=True
)
countries = spark.read.csv(
    'file:/content/gdrive/MyDrive/Big_Data/Pyspark_weather_prediction_pipeline/data/countries.csv',
    header=True, inferSchema=True
)

In [8]:
# Identify capital city stations:
# Join cities with countries where city_name matches capital AND iso3 matches
capital_stations = cities.join(
    countries,
    (cities.iso3 == countries.iso3) & (cities.city_name == countries.capital),
    "inner"
).select(
    cities.station_id,
    cities.city_name
).distinct()

# Filter daily_weather to capital cities only
capital_weather = daily_weather.join(
    capital_stations,
    ["station_id", "city_name"],
    "inner"
)

# Top 10 capital cities by record count
top10_cities = (
    capital_weather
    .groupBy("station_id", "city_name")
    .count()
    .orderBy(F.col("count").desc())
    .limit(10)
)

top10_cities.show(truncate=False)

# Filter to top 10 → top10_data
top10_data = capital_weather.join(
    top10_cities.select("station_id", "city_name"),
    ["station_id", "city_name"],
    "inner"
)
print("top10_data:", top10_data.count())

+----------+---------+-----+
|station_id|city_name|count|
+----------+---------+-----+
|06447     |Brussels |69347|
|11035     |Vienna   |61477|
|02485     |Stockholm|59774|
|14236     |Zagreb   |59084|
|03969     |Dublin   |57154|
|33345     |Kiev     |51816|
|38457     |Tashkent |51379|
|26730     |Vilnius  |50333|
|06680     |Vaduz    |49940|
|37549     |Tbilisi  |49697|
+----------+---------+-----+

top10_data: 560001


In [9]:
# Drop records with missing min_temp_c → top10_data2
top10_data2 = top10_data.filter(F.col("min_temp_c").isNotNull())
print("top10_data2:", top10_data2.count())

top10_data2: 530831


In [10]:
# Join with cities to get latitude, longitude, iso3
# Then join with countries to get region, continent
cities_info = cities.select("station_id", "city_name", "latitude", "longitude", "iso3")
countries_info = countries.select("iso3", "region", "continent")

joined = top10_data2.join(cities_info, ["station_id", "city_name"], "inner")
print("After cities join:", joined.count())

joined = joined.join(countries_info, "iso3", "inner")
print("After countries join:", joined.count())

After cities join: 530831
After countries join: 530831


In [11]:
# Feature engineering: year, month, day, lagged temperature features
joined = (
    joined
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
)

# Window spec for lagged features, partitioned per city
window_spec = Window.partitionBy("station_id", "city_name").orderBy("date")

# Previous day temperatures (lag by 1 day)
joined = joined.withColumn(
    "prev_day_min_temp", F.lag("min_temp_c", 1).over(window_spec)
)
joined = joined.withColumn(
    "prev_day_avg_temp", F.lag("avg_temp_c", 1).over(window_spec)
)
joined = joined.withColumn(
    "prev_day_max_temp", F.lag("max_temp_c", 1).over(window_spec)
)

# Rolling averages (7-day and 30-day) for min_temp_c
window_7d = Window.partitionBy("station_id", "city_name").orderBy("date").rowsBetween(-7, -1)
window_30d = Window.partitionBy("station_id", "city_name").orderBy("date").rowsBetween(-30, -1)

joined = joined.withColumn(
    "rolling_7d_min_temp", F.avg("min_temp_c").over(window_7d)
)
joined = joined.withColumn(
    "rolling_30d_min_temp", F.avg("min_temp_c").over(window_30d)
)

# Temperature trend (difference between yesterday and day before)
joined = joined.withColumn(
    "temp_trend", F.col("prev_day_min_temp") - F.lag("min_temp_c", 2).over(window_spec)
)

# Drop rows where lagged features are null (first ~30 records per city)
joined = joined.filter(
    F.col("prev_day_min_temp").isNotNull() &
    F.col("rolling_7d_min_temp").isNotNull() &
    F.col("rolling_30d_min_temp").isNotNull() &
    F.col("temp_trend").isNotNull()
)
print("After lagged features filter:", joined.count())

# Final column selection - NO same-day temp features (avg_temp_c, max_temp_c) to avoid data leakage
final_data = joined.select(
    "city_name", "date", "season", "latitude", "longitude",
    "region", "continent", "year", "month", "day",
    "prev_day_min_temp", "prev_day_avg_temp", "prev_day_max_temp",
    "rolling_7d_min_temp", "rolling_30d_min_temp", "temp_trend",
    "min_temp_c"
)

# Verify no nulls remain
final_data = final_data.na.drop()
print("final_data:", final_data.count())

# Null check
final_data.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in final_data.columns]
).show()

final_data.show(5, truncate=False)

After lagged features filter: 530811
final_data: 269872
+---------+----+------+--------+---------+------+---------+----+-----+---+-----------------+-----------------+-----------------+-------------------+--------------------+----------+----------+
|city_name|date|season|latitude|longitude|region|continent|year|month|day|prev_day_min_temp|prev_day_avg_temp|prev_day_max_temp|rolling_7d_min_temp|rolling_30d_min_temp|temp_trend|min_temp_c|
+---------+----+------+--------+---------+------+---------+----+-----+---+-----------------+-----------------+-----------------+-------------------+--------------------+----------+----------+
|        0|   0|     0|       0|        0|     0|        0|   0|    0|  0|                0|                0|                0|                  0|                   0|         0|         0|
+---------+----+------+--------+---------+------+---------+----+-----+---+-----------------+-----------------+-----------------+-------------------+--------------------+-------

In [12]:
# Save to CSV and reload
final_data.write.csv(
    'file:/content/gdrive/MyDrive/Big_Data/Pyspark_weather_prediction_pipeline/final_data',
    header=True, mode='overwrite'
)

final_data = spark.read.csv(
    'file:/content/gdrive/MyDrive/Big_Data/Pyspark_weather_prediction_pipeline/final_data',
    header=True, inferSchema=True
)
print("Reloaded final_data:", final_data.count())
final_data.printSchema()

# Train-test split by date
train_data = final_data.filter(F.col("date") < "2010-01-01")
test_data = final_data.filter(F.col("date") >= "2010-01-01")

print("Train:", train_data.count())
print("Test:", test_data.count())

Reloaded final_data: 269872
root
 |-- city_name: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- season: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- region: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- prev_day_min_temp: double (nullable = true)
 |-- prev_day_avg_temp: double (nullable = true)
 |-- prev_day_max_temp: double (nullable = true)
 |-- rolling_7d_min_temp: double (nullable = true)
 |-- rolling_30d_min_temp: double (nullable = true)
 |-- temp_trend: double (nullable = true)
 |-- min_temp_c: double (nullable = true)

Train: 227906
Test: 41966


## 2. Build ML Pipeline

In [13]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# Categorical columns to encode
cat_cols = ["city_name", "season", "region", "continent"]

# Numerical columns to scale - using only lagged/historical features to avoid data leakage
num_cols = [
    "latitude", "longitude", "year", "month", "day",
    "prev_day_min_temp", "prev_day_avg_temp", "prev_day_max_temp",
    "rolling_7d_min_temp", "rolling_30d_min_temp", "temp_trend"
]

# Stage 1: StringIndexer for each categorical column
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep")
    for c in cat_cols
]

# Stage 2: OneHotEncoder for each indexed column
encoder = OneHotEncoder(
    inputCols=[c + "_index" for c in cat_cols],
    outputCols=[c + "_ohe" for c in cat_cols]
)

# Stage 3: Assemble numerical features into a vector, then scale
num_assembler = VectorAssembler(inputCols=num_cols, outputCol="num_features")
scaler = StandardScaler(inputCol="num_features", outputCol="scaled_num_features")

# Stage 4: Assemble all features (scaled numerical + encoded categorical)
final_assembler = VectorAssembler(
    inputCols=["scaled_num_features"] + [c + "_ohe" for c in cat_cols],
    outputCol="features"
)

# Stage 5: Model (no hardcoded hyperparameters - will be tuned)
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="min_temp_c"
)

# Build pipeline
pipeline = Pipeline(stages=indexers + [encoder, num_assembler, scaler, final_assembler, gbt])

# Define hyperparameter grid to search
paramGrid = (ParamGridBuilder()
    .addGrid(gbt.maxIter, [50, 100])
    .addGrid(gbt.maxDepth, [5, 7, 10])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .build()
)

print(f"Total hyperparameter combinations to try: {len(paramGrid)}")

# Evaluator for cross-validation (minimize MAE)
evaluator = RegressionEvaluator(
    labelCol="min_temp_c",
    predictionCol="prediction",
    metricName="mae"
)

# Cross-validator with 3 folds
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2  # Run 2 models in parallel (adjust based on your resources)
)

Total hyperparameter combinations to try: 12


In [14]:
# Run cross-validation to find best hyperparameters
# This will train 12 models × 3 folds = 36 total fits
print("Starting cross-validation... (this may take a while)")

cvModel = crossValidator.fit(train_data)

print("Cross-validation complete!")

Starting cross-validation... (this may take a while)
Cross-validation complete!


In [15]:
# Extract best model and its parameters
bestModel = cvModel.bestModel
bestGBT = bestModel.stages[-1]  # GBT is the last stage in the pipeline

print("=== Best Hyperparameters ===")
print(f"maxIter:  {bestGBT.getMaxIter()}")
print(f"maxDepth: {bestGBT.getMaxDepth()}")
print(f"stepSize: {bestGBT.getStepSize()}")

# Show cross-validation results for all parameter combinations
print("\n=== Cross-Validation Results (MAE for each parameter combination) ===")
cvMetrics = cvModel.avgMetrics
for i, (params, metric) in enumerate(zip(paramGrid, cvMetrics)):
    maxIter = params[gbt.maxIter]
    maxDepth = params[gbt.maxDepth]
    stepSize = params[gbt.stepSize]
    print(f"  maxIter={maxIter:3d}, maxDepth={maxDepth:2d}, stepSize={stepSize:.2f} → MAE: {metric:.4f}")

=== Best Hyperparameters ===
maxIter:  100
maxDepth: 7
stepSize: 0.1

=== Cross-Validation Results (MAE for each parameter combination) ===
  maxIter= 50, maxDepth= 5, stepSize=0.05 → MAE: 1.8089
  maxIter= 50, maxDepth= 5, stepSize=0.10 → MAE: 1.7790
  maxIter= 50, maxDepth= 7, stepSize=0.05 → MAE: 1.7796
  maxIter= 50, maxDepth= 7, stepSize=0.10 → MAE: 1.7670
  maxIter= 50, maxDepth=10, stepSize=0.05 → MAE: 1.7852
  maxIter= 50, maxDepth=10, stepSize=0.10 → MAE: 1.7940
  maxIter=100, maxDepth= 5, stepSize=0.05 → MAE: 1.7794
  maxIter=100, maxDepth= 5, stepSize=0.10 → MAE: 1.7590
  maxIter=100, maxDepth= 7, stepSize=0.05 → MAE: 1.7651
  maxIter=100, maxDepth= 7, stepSize=0.10 → MAE: 1.7585
  maxIter=100, maxDepth=10, stepSize=0.05 → MAE: 1.7847
  maxIter=100, maxDepth=10, stepSize=0.10 → MAE: 1.8060


## 3. Evaluate ML Pipeline

In [16]:
# Use the best model from cross-validation for predictions
train_preds = bestModel.transform(train_data)
test_preds = bestModel.transform(test_data)

In [17]:
# Evaluate with multiple metrics
metrics = ["mae", "rmse", "r2"]

print("=== Final Model Evaluation ===\n")
print(f"Best Hyperparameters: maxIter={bestGBT.getMaxIter()}, maxDepth={bestGBT.getMaxDepth()}, stepSize={bestGBT.getStepSize()}")
print()

for metric in metrics:
    eval = RegressionEvaluator(
        labelCol="min_temp_c",
        predictionCol="prediction",
        metricName=metric
    )
    train_score = eval.evaluate(train_preds)
    test_score = eval.evaluate(test_preds)
    print(f"{metric.upper():4s} - Train: {train_score:.4f}, Test: {test_score:.4f}")

=== Final Model Evaluation ===

Best Hyperparameters: maxIter=100, maxDepth=7, stepSize=0.1

MAE  - Train: 1.6722, Test: 1.7383
RMSE - Train: 2.1908, Test: 2.2390
R2   - Train: 0.9396, Test: 0.9267


## 4. Feature Importance Analysis

In [18]:
# Feature importance from the best GBT model
feature_importances = bestGBT.featureImportances.toArray()

# Build feature names list (numerical + categorical one-hot encoded)
feature_names = num_cols.copy()
# Note: One-hot encoded categorical features are added, but their names are complex
# For simplicity, we'll focus on numerical feature importances

# Create a summary of numerical feature importances
print("=== Feature Importance (Numerical Features) ===\n")
num_importances = feature_importances[:len(num_cols)]
importance_data = sorted(zip(num_cols, num_importances), key=lambda x: x[1], reverse=True)

for name, importance in importance_data:
    bar = "█" * int(importance * 50)
    print(f"{name:25s} {importance:.4f} {bar}")

=== Feature Importance (Numerical Features) ===

prev_day_avg_temp         0.8721 ███████████████████████████████████████████
prev_day_min_temp         0.0177 
year                      0.0161 
prev_day_max_temp         0.0144 
rolling_7d_min_temp       0.0126 
temp_trend                0.0123 
month                     0.0104 
rolling_30d_min_temp      0.0098 
day                       0.0092 
latitude                  0.0077 
longitude                 0.0041 
